# Q-Learning

Juego de tablero con hoyos

In [ ]:
import gym
import numpy as np
import random

# Crear entorno
env = gym.make("FrozenLake-v1", is_slippery=True)

# Parámetros del Q-learning
n_states = env.observation_space.n     # 16 estados
n_actions = env.action_space.n         # 4 acciones (izq, abajo, der, arriba)
q_table = np.zeros((n_states, n_actions))

alpha = 0.8         # tasa de aprendizaje
gamma = 0.95        # factor de descuento
epsilon = 1.0       # exploración inicial
epsilon_min = 0.01
epsilon_decay = 0.995

n_episodes = 5000   # número de episodios de entrenamiento
max_steps = 100     # pasos máximos por episodio

# Entrenamiento
for episode in range(n_episodes):
    state = env.reset()[0]
    done = False
    
    for step in range(max_steps):
        # Política ε-greedy
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()  # explorar
        else:
            action = np.argmax(q_table[state])  # explotar

        next_state, reward, done, _, _ = env.step(action)

        # Actualización Q-learning
        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])
        q_table[state, action] = old_value + alpha * (reward + gamma * next_max - old_value)

        state = next_state
        if done:
            break

    # Decaimiento de epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("✅ Entrenamiento finalizado.")

# Evaluación
n_test_episodes = 100
successes = 0

for _ in range(n_test_episodes):
    state = env.reset()[0]
    done = False
    for _ in range(max_steps):
        action = np.argmax(q_table[state])
        state, reward, done, _, _ = env.step(action)
        if done:
            if reward == 1:
                successes += 1
            break

print(f"🎯 Tasa de éxito tras entrenamiento: {successes / n_test_episodes:.2%}")

In [ ]:
np.set_printoptions(precision=2, suppress=True)
print(q_table)

In [ ]:
import time

def play_agent(env, q_table, episodes=3, delay=1.0):
    actions_map = {
        0: '←', 1: '↓', 2: '→', 3: '↑'
    }

    for ep in range(episodes):
        state = env.reset()[0]
        done = False
        print(f"\n🚀 Episodio {ep + 1}")
        env.render()
        time.sleep(delay)

        while not done:
            action = np.argmax(q_table[state])
            print(f"→ Acción: {actions_map[action]}")
            state, reward, done, _, _ = env.step(action)
            env.render()
            time.sleep(delay)

        if reward == 1:
            print("✅ ¡Éxito! Llegó al objetivo.")
        else:
            print("💥 Cayó en un agujero.")

# Visualizar al agente jugando
play_agent(env, q_table, episodes=5, delay=1.0)

# Taxi

El agente debe recoger a un pasajero y dejarlo en destino. El entorno es una cuadrícula de 5x5.

In [ ]:
import gym
import numpy as np
import random
import time

# Crear entorno
env = gym.make("Taxi-v3", render_mode="ansi")  # Modo texto
n_states = env.observation_space.n
n_actions = env.action_space.n

# Inicializar Q-table
q_table = np.zeros((n_states, n_actions))

# Hiperparámetros
alpha = 0.7
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
n_episodes = 10000
max_steps = 100

# Entrenamiento
for episode in range(n_episodes):
    state = env.reset()[0]
    done = False

    for _ in range(max_steps):
        # Política epsilon-greedy
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_state, reward, done, _, _ = env.step(action)
        
        # Actualización Q-learning
        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])
        q_table[state, action] = old_value + alpha * (reward + gamma * next_max - old_value)

        state = next_state
        if done:
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("✅ Entrenamiento finalizado.")

In [ ]:
successes = 0
n_test_episodes = 100

for _ in range(n_test_episodes):
    state = env.reset()[0]
    done = False

    for _ in range(max_steps):
        action = np.argmax(q_table[state])
        state, reward, done, _, _ = env.step(action)
        if done:
            if reward == 20:  # recompensa por dejar correctamente al pasajero
                successes += 1
            break

print(f"🎯 Tasa de éxito: {successes / n_test_episodes:.2%}")

In [ ]:
def play_agent_taxi(env, q_table, episodes=5, delay=1.0):
    for ep in range(episodes):
        state = env.reset()[0]
        done = False
        print(f"\n🚖 Episodio {ep + 1}")
        print(env.render())
        time.sleep(delay)

        for _ in range(max_steps):
            action = np.argmax(q_table[state])
            state, reward, done, _, _ = env.step(action)
            print(env.render())
            time.sleep(delay)
            if done:
                if reward == 20:
                    print("✅ ¡Pasajero entregado con éxito!")
                else:
                    print("💥 Acción incorrecta.")
                break

# Visualizar 3 episodios
play_agent_taxi(env, q_table, episodes=3, delay=1.0)